# Pilot calibration: tune penalties, then test a held-out workload

Run **Runtime → Run all** in Google Colab. This CPU-only notebook creates a small synthetic pilot, saves its manifest and case fixtures, selects penalty settings on calibration cases, and evaluates the selected setting on separate holdout cases.

No API keys, production data, ERP changes, or external dispatch are involved. These examples use arbitrary weighted penalty units. They are not money, observed labor savings, or ground-truth business value. Monetary interpretation requires independently calibrated units and actual operating evidence.

The holdout is a seeded synthetic check, not evidence of real-world generalization. Both groups use the same generator family. This notebook evaluates implementation behavior and illustrates a leakage-controlled experiment.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops
    needs_install = not hasattr(assumption_ops, "PilotDataset")
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

import json
import platform
from copy import deepcopy
from dataclasses import asdict
from pathlib import Path
from pprint import pprint
from time import perf_counter

import scipy
from assumption_ops import Policy
from assumption_ops.calibration import PenaltyCandidate, calibrate
from assumption_ops.pilot import (
    PilotDataset, PilotEpisode, dataset_summary, load_pilot_dataset, save_pilot_dataset,
)
from assumption_ops.replay_fixtures import synthetic_cases


## 1. Build separate chronological episode groups

Each episode contains an initial operational state and ordered absolute evidence updates. Calibration and holdout have different group identifiers and different traces. Merely renaming a duplicate case would not create an independent holdout; the manifest loader checks normalized trace fingerprints.

Repeated snapshots within an episode concern the same evolving orders. Demand, shortage, and objective cost must not be summed across those snapshots as if each were a new order book.

In [ ]:
calibration_cases = synthetic_cases(order_count=8, seed=1729)
holdout_cases = synthetic_cases(order_count=8, seed=1731)
episodes = tuple(
    PilotEpisode(f"cal-{case.name}", "seed-1729", "calibration", case)
    for case in calibration_cases
) + tuple(
    PilotEpisode(f"hold-{case.name}", "seed-1731", "holdout", case)
    for case in holdout_cases
)
dataset = PilotDataset("synthetic-penalty-pilot", episodes)
calibration_ids = {e.episode_id for e in episodes if e.split == "calibration"}
holdout_ids = {e.episode_id for e in episodes if e.split == "holdout"}
assert calibration_ids.isdisjoint(holdout_ids)
pprint(dataset_summary(dataset))


## 2. Save and reload an inspectable pilot manifest

The saved manifest refers to individual case JSON files. Edit an independent copy to evaluate your own cases. Keep group identities meaningful: related operational episodes belong to the same split.

In [ ]:
pilot_directory = Path("pilot_outputs")
manifest_path = save_pilot_dataset(dataset, pilot_directory)
loaded_dataset = load_pilot_dataset(manifest_path)
assert dataset_summary(loaded_dataset) == dataset_summary(dataset)
print(f"Saved manifest: {manifest_path}")
pprint(dataset_summary(loaded_dataset))


## 3. Separate planner penalties from the fixed business score

Candidates alter what the planner optimizes. Comparing each candidate's own objective would reward changing the scale rather than improving the resulting allocation. Selection therefore uses one independently specified fixed scoring policy for every candidate.

The fixed score includes acquisition cost and penalty-weighted lateness, substitution, disruption, and shortage. All candidate outcomes within a case use the same initial incumbent for disruption accounting. This score expresses a declared preference, not a discovered economic truth.

The three candidate settings below intentionally make different tradeoffs. Choosing the scoring weights is a business judgment that should precede holdout inspection.

In [ ]:
candidates = (
    PenaltyCandidate("service_first", 30, 6, 2, 1000),
    PenaltyCandidate("stability_first", 20, 12, 80, 700),
    PenaltyCandidate("cost_sensitive", 10, 30, 5, 150),
)
scoring_policy = Policy(
    late_penalty=25,
    substitution_penalty=10,
    disruption_penalty=8,
    unfilled_penalty=600,
)
pprint([asdict(candidate) for candidate in candidates])
pprint({"fixed_scoring_policy": asdict(scoring_policy)})


## 4. Select only on calibration, then evaluate baseline and selected on holdout

The selection score is the mean across episodes of final-snapshot fixed score divided by `max(1, requested_units)`. It does not sum repeated snapshots or overweight a case merely because it has more updates. Candidate names break exact ranking ties.

The holdout is evaluated only after selection. It compares the initial baseline policy and selected candidate; it does not rerank all candidates on the holdout.

In [ ]:
dataset_before = deepcopy(asdict(loaded_dataset))
candidates_before = deepcopy([asdict(candidate) for candidate in candidates])
scoring_before = deepcopy(asdict(scoring_policy))
started = perf_counter()
report = calibrate(loaded_dataset, candidates, scoring_policy)
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "scipy": scipy.__version__,
    "wall_seconds": perf_counter() - started,
    "external_dispatch": False,
    "synthetic": True,
}
assert asdict(loaded_dataset) == dataset_before
assert [asdict(candidate) for candidate in candidates] == candidates_before
assert asdict(scoring_policy) == scoring_before
pprint(environment)


## 5. Inspect calibration ranking and objective scale

The ranking uses the fixed business score. Planner objectives remain diagnostic: their weights differ across candidates, so comparing raw candidate objective values is invalid. Examine shortage, on-time service, substitutions, and disruption alongside the fixed score.

In [ ]:
ranking_rows = [
    {
        "candidate": ranking["name"],
        "mean_final_fixed_score_per_requested_unit": (
            ranking["mean_final_fixed_score_per_requested_unit"]
        ),
        "eligible": ranking["eligible"],
    }
    for ranking in report["calibration"]["rankings"]
]
quality_rows = []
for ranking in report["calibration"]["rankings"]:
    ranked_episode_ids = {row["episode_id"] for row in ranking["episode_results"]}
    assert ranked_episode_ids == calibration_ids
    assert ranked_episode_ids.isdisjoint(holdout_ids)
    for episode in ranking["episode_results"]:
        final = episode["final"]
        quality_rows.append({
            "candidate": ranking["name"],
            "episode": episode["episode_id"],
            "own_objective": final["own_objective"],
            "fixed_score": final["fixed_score"],
            "normalized_fixed_score": final["normalized_fixed_score"],
            "fill_rate": final["fill_rate"],
            "late_rate": final["late_rate"],
            "constraints_passed": final["constraints_passed"],
        })
pprint(ranking_rows)
pprint(quality_rows)
print("Selected candidate:", report["selected_candidate"])
pprint(report["selected_penalties"])
assert report["external_dispatch"] is False


## 6. Read final holdout outcomes once

These are final snapshots for separate held-out episodes. The counts are service metrics and the costs are synthetic weighted scores. None establishes measured customer benefit or production readiness. Preserve the selected configuration before opening new holdout data.

In [ ]:
holdout_rows = []
assert {row["episode_id"] for row in report["holdout"]["episodes"]} == holdout_ids
for episode in report["holdout"]["episodes"]:
    for strategy in ("baseline", "selected"):
        final = episode[strategy]["final"]
        holdout_rows.append({
            "episode": episode["episode_id"],
            "strategy": strategy,
            "fixed_score": final["fixed_score"],
            "normalized_fixed_score": final["normalized_fixed_score"],
            "requested_units": final["requested_units"],
            "fill_rate": final["fill_rate"],
            "late_rate": final["late_rate"],
            "constraints_passed": final["constraints_passed"],
        })
pprint(holdout_rows)
pprint({key: value for key, value in report["holdout"].items() if key != "episodes"})


## 7. Export the full experiment

The report retains calibration and holdout results separately, plus provenance and configured weights. Local files let you repeat the run and inspect allocation details without any external writes.

In [ ]:
report_path = pilot_directory / "calibration_report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
(pilot_directory / "environment.json").write_text(
    json.dumps(environment, indent=2) + "\n", encoding="utf-8"
)
print(report_path)
print(manifest_path)


## 8. Load your own manifest

This default loads the generated manifest, so no upload is needed. Replace `own_manifest_path` with your pilot manifest path and inspect the split summary. To rerun, pass this dataset to `calibrate` with weights declared before you inspect holdout outcomes.

For a real pilot, collect actual human review outcomes and operating constraints separately. The framework's declared scoring preferences are not labeled ground truth.

In [ ]:
own_manifest_path = manifest_path
own_dataset = load_pilot_dataset(own_manifest_path)
pprint(dataset_summary(own_dataset))
# To evaluate a genuinely new pilot after declaring scoring weights:
# own_report = calibrate(own_dataset, candidates, scoring_policy)
